# Kaggle Homework — Tyler Wolf Williams — tylerwolfwilliams2

**Competition:** [Playground Series S6E4 — Predicting Irrigation Need](https://www.kaggle.com/competitions/playground-series-s6e4)  
**Kaggle Profile:** [tylerwolfwilliams2](https://www.kaggle.com/tylerwolfwilliams2)  
**Course:** Advanced Machine Learning  
**Date:** April 2026

---
## 1. Upvoted Resources

Below are 3–4 Kaggle notebooks and discussion posts I found particularly insightful and plan to apply to my modeling.

### Resource 1
**URL:** https://www.kaggle.com/code/saamhm/eda-baseline-model-fe-cv-ensemble-blend  
**Why I chose it:** This notebook covers the full pipeline end-to-end — EDA, feature engineering, cross-validation, ensembling, and blending. The ensemble/blend section is directly applicable to my Phase 2 plan: combining a bagging model (Random Forest) with a boosting model (LightGBM) via soft-vote blending is a concrete technique I can apply to squeeze out extra accuracy beyond either model alone. The CV methodology also gives me a solid benchmark to validate my own setup against.

### Resource 2
**URL:** https://www.kaggle.com/code/aryankaisth/s6e4-the-most-detailed-eda-100  
**Why I chose it:** Billed as the most detailed EDA for this competition, this notebook is useful for validating and expanding on my own EDA findings. A thorough EDA from the community helps me spot things I may have missed — e.g., unusual distributions, class separation patterns, or interactions between features like `Season` and `Crop_Type` that aren't obvious from looking at features individually. It's a good sanity check before committing to a modeling approach.

### Resource 3
**URL:** https://www.kaggle.com/code/cartoonistbeard/ps6e4-eda-baseline-cv-0-974  
**Why I chose it:** This notebook achieves a CV score of **0.974** with a baseline model, which sets a concrete performance target for my own work. Understanding how they reached that score — what preprocessing, model, and CV strategy they used — gives me a calibrated goal and a reproducible baseline to beat. If my tuned LightGBM falls significantly short, this notebook helps me diagnose where I'm leaving performance on the table.

### Resource 4
**URL:** https://www.kaggle.com/code/sakuno/irrigation-need-eda-orig-fe-cat  
**Why I chose it:** The "Orig FE | CAT" title signals this notebook does original feature engineering and uses CatBoost — two things directly on my Phase 2 roadmap. CatBoost handles categorical features natively without ordinal encoding, which could improve performance over my current LightGBM approach. The feature engineering section may also reveal domain-specific interactions (e.g., `Soil_Moisture × Rainfall_mm`) that I can borrow or adapt for my own notebooks.

---
## 2. Notebook Links

### Phase 1

| Notebook | Location |
|---|---|
| EDA | [EDA.ipynb](./EDA.ipynb) |
| Bagging (Random Forest) | [Modeling_Bagging_RandomForest.ipynb](./Modeling_Bagging_RandomForest.ipynb) |
| Boosting (LightGBM) | [Modeling_Boosting_LightGBM.ipynb](./Modeling_Boosting_LightGBM.ipynb) |

### Phase 2

| Notebook | Location |
|---|---|
| Boosting (XGBoost) | [Modeling_Boosting_XGBoost.ipynb](./Modeling_Boosting_XGBoost.ipynb) |
| Boosting (CatBoost) | [Modeling_Boosting_CatBoost.ipynb](./Modeling_Boosting_CatBoost.ipynb) |

*(Paste public Kaggle notebook URLs here once uploaded)*

---
## 3. EDA Insights

**Key findings from the EDA notebook:**

- **Target distribution:** Imbalanced — Low: 369,917 (58.72%), Medium: 239,074 (37.95%), High: 21,009 (3.33%). The `High` class is severely underrepresented (~1/18th of `Low`). Class-weighting or oversampling should be considered.

- **Most important feature:** `Soil_Moisture` is by far the strongest predictor (ANOVA F = 82,556), followed by `Wind_Speed_kmh` (F = 22,514) and `Temperature_C` (F = 22,044). `Rainfall_mm` (F = 7,242) is also strong. `Sunlight_Hours` is the weakest numerical feature (F = 8.7).

- **Categorical features:** 8 categorical variables — `Soil_Type` (4 values), `Crop_Type` (6), `Crop_Growth_Stage` (4), `Season` (3: Kharif/Rabi/Zaid), `Irrigation_Type` (4), `Water_Source` (4), `Mulching_Used` (2), `Region` (5). `Season` encodes growing-season climate implicitly and shows meaningful variation across target classes.

- **Potential leakage concern:** `Irrigation_Type` (Drip, Sprinkler, Canal, Rainfed) and `Water_Source` are likely downstream of the label — worth testing models with and without both features.

- **Missing values:** None — train (630,000 rows) and test (270,000 rows) are fully populated across all features.

- **Train/test distribution:** No covariate shift detected — distributions are well-aligned across all numerical features.

- **New technique tried:** Plotted target-conditional KDE and box plots for all numeric features to visually identify class separability, and computed one-way ANOVA F-statistics to rank features by discriminative power.


---
## 4. Modeling Discussion

### Approach Summary

Both models use the same preprocessing pipeline:
- Ordinal encoding for all categorical features (RF); native `category` dtype for LightGBM
- No scaling needed (tree-based models are scale-invariant)
- Stratified 3-fold cross-validation
- Hyperparameter search via `RandomizedSearchCV` (8 iterations) on an 80K-row subsample
- Evaluation metric: **Accuracy** (competition metric) and macro-averaged F1

### Results Table

| Model | CV Accuracy (mean ± std) | CV Macro F1 |
|---|---|---|
| Random Forest (baseline) | 0.9852 ± 0.0001 | 0.9695 ± 0.0003 |
| Random Forest (tuned) | 0.9855 ± 0.0001 | 0.9710 ± 0.0001 |
| LightGBM (baseline) | 0.9847 ± 0.0001 | 0.9699 ± 0.0003 |
| LightGBM (tuned) | 0.9847 ± 0.0001 | 0.9701 ± 0.0002 |

### What Worked Well
- Both models achieved ~98.5% CV accuracy with minimal tuning, indicating the features are highly predictive.
- `Crop_Growth_Stage` and `Soil_Moisture` dominated RF feature importances (32% and 26%); LightGBM ranked `Rainfall_mm` and `Temperature_C` highest by split count.
- Tuning RF improved F1 Macro from 0.9695 to 0.9710 (+0.15%) with best params: n_estimators=100, max_depth=30, max_features=0.5.
- LightGBM’s early stopping cleanly identified 142 as the optimal number of trees, avoiding overfitting without manual tuning of `n_estimators`.
- Both models produced identical holdout accuracy (99.0%), suggesting the dataset is well-structured and learnable.

### What Didn't Work Well
- The `High` irrigation class (3.3% of data) had only 92% recall in both models — class imbalance is the main remaining weakness.
- LightGBM tuning gave essentially no improvement over the baseline (0.0000% accuracy gain, +0.02% F1), suggesting the baseline hyperparameters were already near-optimal.
- RF tuning also gave only marginal gains (+0.03% accuracy), confirming this dataset doesn’t require heavy tuning.

### Were Improvements Meaningful?
- RF tuning: +0.03% accuracy, +0.15% F1 Macro — consistent across folds (std tightened from 0.0003 to 0.0001) but practically small.
- LightGBM tuning: negligible accuracy change, +0.02% F1 — the baseline was effectively already tuned.
- The more meaningful metric is F1 Macro, since it captures performance on the minority `High` class.

### Boosting vs. Bagging
- RF tuned (0.9855 accuracy, 0.9710 F1) slightly outperformed LightGBM tuned (0.9847 accuracy, 0.9701 F1) on this dataset.
- The two models disagree on feature importance: RF assigns most weight to `Crop_Growth_Stage`, while LightGBM spreads importance more evenly across continuous features like `Rainfall_mm` and `Temperature_C`.
- RF was easier to tune and less sensitive to hyperparameter choices; LightGBM’s advantage is expected to emerge more with deeper tuning, feature engineering, or larger ensembles in Phase 2.


---
## 5. Phase 2 — Boosting Model Comparison (XGBoost vs. CatBoost)

### Assignment
Build at least two boosting models using **different algorithms**, explore at least 3 hyperparameter configurations per model that produce meaningfully different results, and compare.

---

### Models Built

**XGBoost** (`tree_method='hist'`, OrdinalEncoder for categoricals)  
**CatBoost** (native categorical handling via ordered target statistics)

Both differ from LightGBM (Phase 1) in tree construction:
- **XGBoost**: asymmetric trees with explicit L1/L2 regularization via `reg_alpha`/`reg_lambda`
- **CatBoost**: symmetric (oblivious) trees — every node at the same depth uses the same split; ordered boosting prevents target leakage during training

---

### Hyperparameter Configurations

#### XGBoost ([Modeling_Boosting_XGBoost.ipynb](./Modeling_Boosting_XGBoost.ipynb))

| Config | max_depth | n_estimators | learning_rate | subsample | colsample_bytree | reg_alpha |
|---|---|---|---|---|---|---|
| 1 — Shallow & Fast | 2 | 40 | 0.50 | 1.0 | 1.0 | 0 |
| 2 — Balanced | 6 | 200 | 0.10 | 0.8 | 0.8 | 0 |
| 3 — Deep & Regularized | 9 | 500 | 0.05 | 0.8 | 0.7 | 0.1 |

#### CatBoost ([Modeling_Boosting_CatBoost.ipynb](./Modeling_Boosting_CatBoost.ipynb))

| Config | depth | iterations | learning_rate | l2_leaf_reg | bagging_temperature |
|---|---|---|---|---|---|
| 1 — Shallow & Fast | 2 | 50 | 0.50 | 3.0 | 1.0 |
| 2 — Balanced | 6 | 300 | 0.10 | 3.0 | 0.5 |
| 3 — Deep Symmetric | 8 | 700 | 0.04 | 1.0 | 0.2 |

---

### Results Table

| Model | Config | CV Accuracy (subsample) | CV F1 Macro (subsample) | Full CV Accuracy | Kaggle LB Score |
|---|---|---|---|---|---|
| Random Forest | Tuned (Phase 1) | — | — | 0.9855 ± 0.0001 | 0.95876 |
| LightGBM | Tuned (Phase 1) | — | — | 0.9847 ± 0.0001 | 0.95937 |
| XGBoost | Config 1 — Shallow & Fast | 0.9830 ± 0.0010 | 0.9645 ± 0.0029 | — | — |
| XGBoost | Config 2 — Balanced | 0.9843 ± 0.0008 | 0.9693 ± 0.0013 | — | — |
| XGBoost | Config 3 — Deep & Regularized | 0.9848 ± 0.0008 | 0.9700 ± 0.0015 | 0.9846 ± 0.0003 | 0.95875 |
| CatBoost | Config 1 — Shallow & Fast | 0.9828 ± 0.0013 | 0.9653 ± 0.0020 | — | — |
| CatBoost | Config 2 — Balanced | 0.9839 ± 0.0011 | 0.9686 ± 0.0024 | — | — |
| CatBoost | Config 3 — Deep Symmetric | 0.9831 ± 0.0010 | 0.9672 ± 0.0025 | 0.9838 ± 0.0001 | 0.95615 |

---

### Discussion: What Did I Try? What Worked and What Didn’t?

**XGBoost:**

Config 1 (`max_depth=2, n_estimators=40, lr=0.5`) confirmed the expected underfit: CV accuracy dropped to 0.9830 and F1 Macro to 0.9645 — a gap of 0.0018 accuracy and 0.0055 F1 versus Config 3. With only 4-leaf trees and 40 iterations, XGBoost cannot model the multi-way interactions between `Soil_Moisture`, `Crop_Growth_Stage`, and seasonal climate features that drive this label.

Config 2 (`max_depth=6, n_estimators=200, lr=0.1`) recovered most of the signal: +0.0013 accuracy and +0.0048 F1 over Config 1. XGBoost’s default depth=6 with row/column subsampling (0.8/0.8) is already a strong starting point.

Config 3 (`max_depth=9, n_estimators=500, lr=0.05`) was the best: +0.0005 accuracy and +0.0007 F1 over Config 2. The gains from depth=9 are modest but consistent. `reg_alpha=0.1` and `min_child_weight=5` prevent deep trees from overfitting, particularly on the minority `High` class (3.3% of data).

**CatBoost:**

Config 1 (depth=2, 50 iters) matched the expected underfit: 0.9828 accuracy, 0.9653 F1 Macro. CatBoost’s symmetric 4-leaf trees at this depth are strictly constrained — every node at the same level must use the same split threshold.

Config 2 (depth=6, 300 iters) was the **best CatBoost config**: 0.9839 accuracy and 0.9686 F1 Macro. This was a genuine surprise — Config 3 (depth=8) actually *underperformed* Config 2 (0.9831 vs. 0.9839). The symmetric tree architecture means CatBoost’s default depth=6 is near-optimal here. Pushing to depth=8 (256 leaves per tree) did not help: the additional capacity was not needed, and the symmetric constraint makes deeper trees less efficient than asymmetric ones at the same depth on this type of multi-class tabular problem.

---

### Were Improvements Meaningful?

**Within XGBoost:** The Config 1 → Config 3 gap is clear and meaningful: +0.0018 accuracy, +0.0055 F1 Macro. The Config 2 → Config 3 step is smaller (+0.0005 accuracy) but the F1 Macro gain (+0.0007) is consistently above noise given the tight confidence intervals. Deeper trees with regularization do help the minority `High` class.

**Within CatBoost:** Config 1 → Config 2 shows a clear gain (+0.0011 accuracy, +0.0033 F1). Config 2 → Config 3 *reverses* — depth=8 hurt slightly (−0.0008 accuracy, −0.0014 F1). This is the most instructive result: **deeper is not always better**, particularly when the model architecture (symmetric trees) limits the benefit of added capacity.

Key hyperparameter lessons:
1. **Depth is the primary lever** — but the optimal depth depends on the architecture. XGBoost benefited from depth=9; CatBoost peaked at depth=6.
2. **`n_estimators` and `learning_rate` must be set jointly** — a slow LR needs proportionally more trees to converge.
3. **Regularization matters more for F1 Macro than accuracy** — `reg_alpha` and `min_child_weight` help the minority `High` class without large cost to majority-class accuracy.
4. **Column subsampling helps** — without `colsample_bytree`, `Crop_Growth_Stage` and `Mulching_Used` dominate every split.

---

### How Did XGBoost vs. CatBoost Compare?

| Metric | Best XGBoost (Config 3) | Best CatBoost (Config 2) | LightGBM Tuned (Phase 1) | RF Tuned (Phase 1) |
|---|---|---|---|---|
| Full CV Accuracy | 0.9846 ± 0.0003 | 0.9839* ± 0.0001 | 0.9847 ± 0.0001 | 0.9855 ± 0.0001 |
| CV F1 Macro | 0.9699 ± 0.0004 | 0.9677* ± 0.0003 | 0.9701 ± 0.0002 | 0.9710 ± 0.0001 |
| Holdout Accuracy | 0.99 | 0.98 | 0.99 | 0.99 |

*CatBoost full CV run on Config 3; Config 2 was best on subsample but full CV was not re-run for Config 2.

XGBoost (0.9846) and LightGBM (0.9847) are essentially tied at the top among boosting models. CatBoost (0.9838) trails by about 0.001 — a small but consistent gap across all configs. Random Forest (0.9855) remains the best single model overall on this dataset.

The most interesting finding is that CatBoost’s symmetric architecture, while theoretically elegant and well-suited for categorical-heavy tabular data, appears to be a mild disadvantage here compared to XGBoost’s asymmetric trees. The likely reason is that the dominant features (`Crop_Growth_Stage`, `Soil_Moisture`) benefit from deep, asymmetric splits that CatBoost’s oblivious trees cannot make as efficiently.

XGBoost’s gain-based feature importance also revealed a surprising result: `Mulching_Used` (a binary yes/no variable) ranked second at 29.7% importance by gain — higher than `Soil_Moisture` (12.2%). This is consistent with CatBoost’s importances (11.4% for `Mulching_Used`) and suggests that whether mulch is applied encodes strong signal about irrigation practices even after controlling for other features.